# Adult income prediction

DSDA 310 Project 1

This notebook compares logistic regression with a random forest, then looks at the model's mistakes and group differences. It covers the work through the September 29 milestone. See [README.md](README.md) for setup and [the proposal](01_proposal.md) for the feature choices. Run the cells in order from the project folder.

## 1. Run the analysis

The model predicts whether income is above $50K. The code in [analysis.py](analysis.py) loads the data, fits the models, and saves the results. Missing values and categories are handled using training data only. Forest depth is chosen on validation data before the separate test file is scored.

In [1]:
from pathlib import Path
from analysis import run
ROOT = Path.cwd()
assert (ROOT / "DSDA310" / "adult.data").is_file(), "Run this notebook from 310.Exercise1."
result = run(ROOT)

Training 26,049; validation 6,512; official test 16,281
Fitted Majority class
Fitted Logistic regression
Forest depth=12: validation F1=0.6772
Forest depth=20: validation F1=0.7017
Forest depth=None: validation F1=0.7058
Comparison and test predictions saved.
Checked Full features
Checked Without capital gains and losses
Checked Without sex and race
Computing global permutation importance on validation data...
All checks passed. Reports and explanation artifacts generated.


## 2. Check the data

The table below shows missing values. Numeric fields use the training median; categories use the training mode. No rows are dropped. Identical records stay together when the training and validation sets are split.

In [2]:
import pandas as pd
print(pd.read_csv(ROOT / "outputs" / "missing_values.csv").to_string(index=False))
print({k: result["manifest"][k] for k in ["training_rows", "validation_rows", "test_rows", "overlapping_test_predictor_rows"]})

        column  development  test
           age            0     0
     workclass         1836   963
        fnlwgt            0     0
     education            0     0
 education_num            0     0
marital_status            0     0
    occupation         1843   966
  relationship            0     0
          race            0     0
           sex            0     0
  capital_gain            0     0
  capital_loss            0     0
hours_per_week            0     0
native_country          583   274
        income            0     0
{'training_rows': 26049, 'validation_rows': 6512, 'test_rows': 16281, 'overlapping_test_predictor_rows': 26}


## 3. Compare the models

F1 is the main metric because higher-income records are less common. The majority-class model shows why accuracy alone can be misleading. Logistic regression is the baseline; the random forest can capture more complicated patterns. Both use the same split and features. These results give each record equal weight.

In [3]:
print(result["metrics"].round(4).to_string(index=False))
print("\nValidation-only forest search:")
print(pd.read_csv(ROOT / "outputs" / "validation_search.csv").round(4).to_string(index=False))

              model      split     n  accuracy  precision  recall     f1  roc_auc  average_precision    tn   fp   fn   tp
     Majority class      train 26049    0.7592     0.0000  0.0000 0.0000   0.5000             0.2408 19776    0 6273    0
     Majority class validation  6512    0.7592     0.0000  0.0000 0.0000   0.5000             0.2408  4944    0 1568    0
     Majority class       test 16281    0.7638     0.0000  0.0000 0.0000   0.5000             0.2362 12435    0 3846    0
Logistic regression      train 26049    0.8489     0.7297  0.5916 0.6534   0.9041             0.7591 18401 1375 2562 3711
Logistic regression validation  6512    0.8606     0.7531  0.6263 0.6838   0.9129             0.7846  4622  322  586  982
Logistic regression       test 16281    0.8487     0.7204  0.5876 0.6473   0.9025             0.7552 11558  877 1586 2260
      Random forest      train 26049    0.8791     0.8281  0.6282 0.7145   0.9434             0.8524 18958  818 2332 3941
      Random forest vali

## 4. Which features matter overall?

Each feature is shuffled on the validation set five times. A larger drop in F1 means the model relies more on that feature. Related features can share importance, so this does not tell us what causes income to change. The table and interpretation are in [the explanation report](explainability/README.md).

In [4]:
print(result["importance"].round(5).to_string(index=False))

       feature  mean_f1_decrease  std_f1_decrease
marital_status           0.11989          0.00669
  relationship           0.11344          0.00533
 education_num           0.09222          0.00620
  capital_gain           0.09167          0.00281
    occupation           0.06147          0.00185
           age           0.05184          0.00734
hours_per_week           0.03151          0.00304
  capital_loss           0.02269          0.00184
           sex           0.01984          0.00373
     workclass           0.01431          0.00311
native_country           0.00237          0.00188
          race           0.00148          0.00091


## 5. Look at one wrong prediction

The example is the first test record the forest gets wrong. The contributions show how each feature raises or lowers its predicted probability as the record moves through the trees. They add up to the final prediction when combined with the starting probability. This is a tree-path explanation, not SHAP. Read the case discussion in [the explanation report](explainability/README.md).

In [5]:
import json
print(json.dumps(result["case"], indent=2))
print(result["local"].round(5).to_string(index=False))

{
  "test_record": 3,
  "physical_file_line": 4,
  "true_label": 1,
  "predicted_label": 0,
  "probability": 0.40359170846762615,
  "baseline_probability": 0.2409445660102115,
  "threshold": 0.5,
  "features": {
    "age": 28,
    "workclass": "Local-gov",
    "fnlwgt": 336951,
    "education": "Assoc-acdm",
    "education_num": 12,
    "marital_status": "Married-civ-spouse",
    "occupation": "Protective-serv",
    "relationship": "Husband",
    "race": "White",
    "sex": "Male",
    "capital_gain": 0,
    "capital_loss": 0,
    "hours_per_week": 40,
    "native_country": "United-States",
    "income": ">50K"
  }
}
       feature  contribution              value
           age      -0.10842                 28
marital_status       0.09789 Married-civ-spouse
  relationship       0.07300            Husband
     workclass       0.06953          Local-gov
    occupation       0.05181    Protective-serv
  capital_gain      -0.02459                  0
hours_per_week      -0.01703           

## 6. Check leakage and group differences

First, remove capital gains and losses to see how much the model depends on them. Those values may not be available if income is predicted before year-end. Next, compare errors across sex and race groups and retrain without those two fields. Other features can still carry related information. The full interpretation is in [the evaluation report](fairness_leakage_check.md).

In [6]:
print(result["ablations"].round(4).to_string(index=False))
print("\nGroup metrics:")
print(result["groups"].round(4).to_string(index=False))
print("\nOverlap sensitivity:")
print(pd.read_csv(ROOT / "outputs" / "overlap_sensitivity.csv").round(4).to_string(index=False))

                           model      split     n  accuracy  precision  recall     f1  roc_auc  average_precision    tn  fp   fn   tp
                   Full features validation  6512    0.8745     0.8106  0.6250 0.7058   0.9258             0.8216  4715 229  588  980
                   Full features       test 16281    0.8649     0.7839  0.5913 0.6741   0.9157             0.7979 11808 627 1572 2274
Without capital gains and losses validation  6512    0.8521     0.7494  0.5797 0.6537   0.9003             0.7306  4640 304  659  909
Without capital gains and losses       test 16281    0.8385     0.7103  0.5343 0.6099   0.8905             0.7091 11597 838 1791 2055
            Without sex and race validation  6512    0.8750     0.8060  0.6333 0.7093   0.9251             0.8230  4705 239  575  993
            Without sex and race       test 16281    0.8647     0.7814  0.5931 0.6744   0.9160             0.8002 11797 638 1565 2281

Group metrics:
               model attribute              gr

## 7. Check the results

The checks below recalculate scores from saved predictions, check the split and group counts, and confirm that the local explanation adds up correctly. The [executive summary](executive_summary.md) gives the main findings in plain language. The [AI-use log](ai_use_log.md) records the assistance and checks used.

In [7]:
from verify_results import verify
verify(ROOT)

PASS: exported metrics, confusion matrices, group rates, split separation, failure-case reconstruction and threshold-tie consistency.
